In [ ]:
# Python API analyzer

import ast
from pathlib import Path
from collections import defaultdict


def _annotate_parents(tree: ast.AST) -> None:
    """Add `.parent` links so we can walk up the tree."""
    for parent in ast.walk(tree):
        for child in ast.iter_child_nodes(parent):
            child.parent = parent


def _in_type_checking_block(node: ast.AST) -> bool:
    """Check if in an `if TYPE_CHECKING:` block."""
    while hasattr(node, "parent"):
        parent = node.parent
        if isinstance(parent, ast.If):
            if isinstance(parent.test, ast.Name) and parent.test.id == "TYPE_CHECKING":
                return True
        node = parent
    return False


class PmgAnalyzerPy(ast.NodeVisitor):
    """Python analyzer for pymatgen."""

    def __init__(self) -> None:
        # alias map: local name → full path
        self.aliases: dict[str, str] = {}
        # usage counts
        self.usage: dict[str, int] = defaultdict(int)

    def visit_Import(self, node) -> None:
        if _in_type_checking_block(node):
            return
        for alias in node.names:
            if alias.name.startswith("pymatgen"):
                asname = alias.asname or alias.name.split(".")[-1]
                self.aliases[asname] = alias.name

    def visit_ImportFrom(self, node) -> None:
        if _in_type_checking_block(node):
            return
        if node.module and node.module.startswith("pymatgen"):
            for alias in node.names:
                asname = alias.asname or alias.name
                self.aliases[asname] = f"{node.module}.{alias.name}"

    def visit_Call(self, node) -> None:
        """Track function/method calls on pymatgen objects."""
        if isinstance(node.func, ast.Attribute):
            base = node.func.value
            if isinstance(base, ast.Name) and base.id in self.aliases:
                full = f"{self.aliases[base.id]}.{node.func.attr}"
                self.usage[full] += 1
        self.generic_visit(node)


def analyze_py(path: str | Path) -> tuple[dict[str, str], dict[str, int]]:
    """
    Analyze a Python file for pymatgen API usage.

    Returns:
        aliases (dict): Mapping of local names → full pymatgen paths.
        usage (dict): Mapping of pymatgen API calls → count.
    """
    path = Path(path)

    if path.suffix != ".py":
        raise ValueError(f"cannot analyze non-py file: {path}")

    text = path.read_text(encoding="utf-8")

    tree = ast.parse(text)

    _annotate_parents(tree)

    analyzer = PmgAnalyzerPy()
    analyzer.visit(tree)
    return analyzer.aliases, dict(analyzer.usage)

In [ ]:
# ipynb API analyzer

import json
from pathlib import Path
from tempfile import NamedTemporaryFile


def analyze_notebook(path: str | Path) -> tuple[dict[str, str], dict[str, int]]:
    path = Path(path)
    if path.suffix != ".ipynb":
        raise ValueError(f"cannot analyze non-ipynb file: {path}")

    nb = json.loads(path.read_text(encoding="utf-8"))
    aliases, usage = {}, {}

    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        code = "".join(cell.get("source", []))

        # Write code to a temporary .py file
        with NamedTemporaryFile("w", suffix=".py", delete=False) as tmp:
            tmp.write(code)
            tmp_path = Path(tmp.name)

        _alias, _usage = analyze_py(tmp_path)

        # Merge results
        aliases.update(_alias)
        for key, val in _usage.items():
            usage[key] = usage.get(key, 0) + val

        tmp_path.unlink(missing_ok=False)

    return aliases, usage

In [ ]:
"""Sanity check with .py and .ipynb files."""
test_py_file = "./dependent-repos/pymatviz/pymatviz/chem_env.py"
aliases, usage = analyze_py(test_py_file)

print(aliases)
print(usage)

test_ipynb_file = "/Users/yang/developer/pymatgen-2-paper/figs/dependents-of-pmg/dependent-repos/pymatviz/examples/widgets/jupyter_demo.ipynb"
aliases, usage = analyze_notebook(test_ipynb_file)

print(aliases)
print(usage)

{'local_env': 'pymatgen.analysis.local_env'}
{'pymatgen.analysis.local_env.LocalStructOrderParams': 1, 'pymatgen.analysis.local_env.CrystalNN': 1}
{'Composition': 'pymatgen.core.Composition', 'Lattice': 'pymatgen.core.Lattice', 'Structure': 'pymatgen.core.Structure'}
{'pymatgen.core.Lattice.cubic': 1}
